In [1]:
!pip install datasets pandas langchain chromadb openai tiktoken load_dotenv

In [2]:
import os
import json
import ast
import pandas as pd

from datasets import load_dataset

from langchain_text_splitters  import RecursiveCharacterTextSplitter

from openai import OpenAI

import chromadb

from dotenv import load_dotenv
load_dotenv()

True

In [3]:
ds = load_dataset("allenai/qasper")

dfs = []

for split_name in ds.keys():

    df = ds[split_name].to_pandas()

    df["split"] = split_name

    dfs.append(df)

merged_df = pd.concat(dfs, ignore_index=True)

print("Total rows:", len(merged_df))

merged_df.head()

Total rows: 1585


,id,title,abstract,full_text,qas,figures_and_tables,split
0,1909.00694,Minimally Supervised Learning of Affective Eve...,Recognizing affective events that trigger posi...,"{'section_name': ['Introduction', 'Related Wor...","{'question': ['What is the seed lexicon?', 'Wh...",{'caption': ['Figure 1: An overview of our met...,train
1,2003.07723,"PO-EMO: Conceptualization, Annotation, and Mod...",Most approaches to emotion analysis regarding ...,"{'section_name': ['', ' ::: ', ' ::: ::: ', '...",{'question': ['Does the paper report macro F1?...,{'caption': ['Figure 1: Temporal distribution ...,train
2,1705.09665,Community Identity and User Engagement in a Mu...,A community's identity defines and shapes its ...,"{'section_name': ['Introduction', 'A typology ...",{'question': ['Do they report results only on ...,{'caption': ['Figure 1: A: Within a community ...,train
3,1908.06606,Question Answering based Clinical Text Structu...,Clinical text structuring is a critical and fu...,"{'section_name': ['Introduction', 'Related Wor...",{'question': ['What data is the language model...,{'caption': ['Fig. 1. An illustrative example ...,train
4,1811.00942,Progress and Tradeoffs in Neural Language Models,"In recent years, we have witnessed a dramatic ...","{'section_name': ['Introduction', 'Background ...",{'question': ['What aspects have been compared...,{'caption': ['Table 1: Comparison of neural la...,train


In [4]:
# Create dataframe subsets from merged_df

df_50 = merged_df.iloc[:50].copy()
df_100 = merged_df.iloc[:100].copy()
df_250 = merged_df.iloc[:250].copy()
df_500 = merged_df.iloc[:500].copy()
df_750 = merged_df.iloc[:750].copy()
df_1000 = merged_df.iloc[:1000].copy()
df_1500 = merged_df.iloc[:1500].copy()

# Print lengths of all subsets

print("Length of df_50   :", len(df_50))
print("Length of df_100  :", len(df_100))
print("Length of df_250  :", len(df_250))
print("Length of df_500  :", len(df_500))
print("Length of df_750  :", len(df_750))
print("Length of df_1000 :", len(df_1000))
print("Length of df_1500 :", len(df_1500))

Length of df_50   : 50
Length of df_100  : 100
Length of df_250  : 250
Length of df_500  : 500
Length of df_750  : 750
Length of df_1000 : 1000
Length of df_1500 : 1500


In [5]:
base_path = "benchmark"

folders = [
    "data",
    "rag",
    "wikillm",
    "evaluation"
]

for folder in folders:
    os.makedirs(f"{base_path}/{folder}", exist_ok=True)

scales = ["docs_50", "docs_100", "docs_250", "docs_500", "docs_750", "docs_1000", "docs_1500"]

for scale in scales:

    # RAG folders
    os.makedirs(f"{base_path}/rag/{scale}/chroma_db", exist_ok=True)

    # WikiLLM folders
    os.makedirs(f"{base_path}/wikillm/{scale}/pages", exist_ok=True)

print("Folders created")

Folders created


In [6]:
def build_documents(df):

    docs = []

    for idx, row in df.iterrows():

        text = f"""
TITLE:
{row['title']}

ABSTRACT:
{row['abstract']}

FULL TEXT:
{row['full_text']}
"""

        docs.append({
            "doc_id": str(idx),
            "text": text
        })

    return docs


docs_50 = build_documents(df_50)
docs_100 = build_documents(df_100)
docs_250 = build_documents(df_250)
docs_500 = build_documents(df_500)
docs_750 = build_documents(df_750)
docs_1000 = build_documents(df_1000)
docs_1500 = build_documents(df_1500)

print(len(docs_50))
print(len(docs_750))

50
750


In [7]:
docs_100[0]

{'doc_id': '0',
 'text': '\nTITLE:\nMinimally Supervised Learning of Affective Events Using Discourse Relations\n\nABSTRACT:\nRecognizing affective events that trigger positive or negative sentiment has a wide range of natural language processing applications but remains a challenging problem mainly because the polarity of an event is not necessarily predictable from its constituent words. In this paper, we propose to propagate affective polarity using discourse relations. Our method is simple and only requires a very small seed lexicon and a large raw corpus. Our experiments using Japanese data show that our method learns affective events effectively without manually labeled data. It also improves supervised learning results when labeled data are small.\n\nFULL TEXT:\n{\'section_name\': array([\'Introduction\', \'Related Work\', \'Proposed Method\',\n       \'Proposed Method ::: Polarity Function\',\n       \'Proposed Method ::: Discourse Relation-Based Event Pairs\',\n       \'Propos

In [8]:
print("Total characters in each document scale:\n")

print("docs_50    ->", format(sum(len(doc["text"]) for doc in docs_50), ","))
print("docs_100   ->", format(sum(len(doc["text"]) for doc in docs_100), ","))
print("docs_250   ->", format(sum(len(doc["text"]) for doc in docs_250), ","))
print("docs_500   ->", format(sum(len(doc["text"]) for doc in docs_500), ","))
print("docs_750   ->", format(sum(len(doc["text"]) for doc in docs_750), ","))
print("docs_1000  ->", format(sum(len(doc["text"]) for doc in docs_1000), ","))
print("docs_1500  ->", format(sum(len(doc["text"]) for doc in docs_1500), ","))

Total characters in each document scale:

docs_50    -> 1,827,875
docs_100   -> 3,614,544
docs_250   -> 9,299,936
docs_500   -> 18,905,158
docs_750   -> 28,074,999
docs_1000  -> 37,012,563
docs_1500  -> 54,338,157


In [9]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1024,
    chunk_overlap=20
)

In [10]:
def create_chunks(docs):

    chunks = []

    for doc in docs:

        split_texts = splitter.split_text(doc["text"])

        for i, chunk in enumerate(split_texts):

            chunks.append({
                "chunk_id": f"{doc['doc_id']}_{i}",
                "doc_id": doc["doc_id"],
                "text": chunk
            })

    return chunks

# Create chunks for all document scales

chunks_50 = create_chunks(docs_50)
chunks_100 = create_chunks(docs_100)
chunks_250 = create_chunks(docs_250)
chunks_500 = create_chunks(docs_500)
chunks_750 = create_chunks(docs_750)
chunks_1000 = create_chunks(docs_1000)
chunks_1500 = create_chunks(docs_1500)


# Print chunk counts

print("50 docs chunks   :", len(chunks_50))
print("100 docs chunks  :", len(chunks_100))
print("250 docs chunks  :", len(chunks_250))
print("500 docs chunks  :", len(chunks_500))
print("750 docs chunks  :", len(chunks_750))
print("1000 docs chunks :", len(chunks_1000))
print("1500 docs chunks :", len(chunks_1500))

50 docs chunks   : 2576
100 docs chunks  : 5084
250 docs chunks  : 13074
500 docs chunks  : 26551
750 docs chunks  : 39551
1000 docs chunks : 52097
1500 docs chunks : 76461


In [11]:
# Save chunks for all document scales

with open(f"{base_path}/rag/docs_50/chunks.json", "w") as f:
    json.dump(chunks_50, f)

with open(f"{base_path}/rag/docs_100/chunks.json", "w") as f:
    json.dump(chunks_100, f)

with open(f"{base_path}/rag/docs_250/chunks.json", "w") as f:
    json.dump(chunks_250, f)

with open(f"{base_path}/rag/docs_500/chunks.json", "w") as f:
    json.dump(chunks_500, f)

with open(f"{base_path}/rag/docs_750/chunks.json", "w") as f:
    json.dump(chunks_750, f)

with open(f"{base_path}/rag/docs_1000/chunks.json", "w") as f:
    json.dump(chunks_1000, f)

with open(f"{base_path}/rag/docs_1500/chunks.json", "w") as f:
    json.dump(chunks_1500, f)

print("All chunks saved successfully")

All chunks saved successfully


In [12]:
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

In [13]:
def get_embedding(texts):

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )

    return [item.embedding for item in response.data]

In [14]:
texts = [
    "This is first chunk",
    "This is second chunk",
    "This is third chunk"
]

embeddings = get_embedding(texts)

print(len(embeddings))

3


In [15]:
embeddings[0]==embeddings[1]

False

In [16]:
# Batch embedding function

def get_embedding(texts):

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )

    return [item.embedding for item in response.data]


# Build Chroma DB with batch processing

def build_chroma_db(chunks, db_path, batch_size=50):

    chroma_client = chromadb.PersistentClient(path=db_path)

    collection = chroma_client.get_or_create_collection(
        name="rag_collection"
    )

    total_batches = (len(chunks) + batch_size - 1) // batch_size

    for i in range(0, len(chunks), batch_size):

        try:

            batch_chunks = chunks[i:i + batch_size]

            texts = [
                chunk["text"]
                for chunk in batch_chunks
            ]

            ids = [
                chunk["chunk_id"]
                for chunk in batch_chunks
            ]

            metadatas = [
                {
                    "doc_id": str(chunk["doc_id"])
                }
                for chunk in batch_chunks
            ]

            # Batch embedding API call
            embeddings = get_embedding(texts)

            # Store in ChromaDB
            collection.add(
                ids=ids,
                documents=texts,
                embeddings=embeddings,
                metadatas=metadatas
            )

            current_batch = (i // batch_size) + 1

            #print(
                #f"Processed batch "
                #f"{current_batch}/{total_batches}"
            #)

        except Exception as e:

            print(
                f"Error in batch "
                f"{(i // batch_size) + 1}: {e}"
            )

    return collection


# Build vector databases

collection_50 = build_chroma_db(
    chunks_50,
    f"{base_path}/rag/docs_50/chroma_db"
)



In [17]:
# Search function

def search_chroma(collection, query, top_k=5):

    # Embed query
    query_embedding = get_embedding([query])[0]

    # Search
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    return results


# Example query
query = "PTSD Detection System"

results = search_chroma(
    collection_50,
    query,
    top_k=5
)

In [18]:
results['documents']

[["array(['Fig. FIGREF7 shows a schematic representation of our proposed model. It consists of the following logical steps: (i) Develop PTSD Detection System using twitter posts of war-veterans(ii) design real surveys from the popular symptoms based mental disease assessment surveys; (iii) define single category and create PTSD Linguistic Dictionary for each survey question and multiple aspect/words for each question; (iv) calculate $\\\\alpha $-scores for each category and dimension based on linguistic inquiry and word count as well as the aspects/words based dictionary; (v) calculate scaling scores ($s$-scores) for each dimension based on the $\\\\alpha $-scores and $s$-scores of each category based on the $s$-scores of its dimensions; (vi) rank features according to the contributions of achieving separation among categories associated with different $\\\\alpha $-scores and $s$-scores; and select feature sets that minimize the overlap among categories as associated with the target cl

In [19]:
collection_100 = build_chroma_db(
    chunks_100,
    f"{base_path}/rag/docs_100/chroma_db"
)
print("docs_100 Vector DB created successfully")


collection_250 = build_chroma_db(
    chunks_250,
    f"{base_path}/rag/docs_250/chroma_db"
)
print("docs_250 Vector DB created successfully")


collection_500 = build_chroma_db(
    chunks_500,
    f"{base_path}/rag/docs_500/chroma_db"
)
print("docs_500 Vector DB created successfully")


collection_750 = build_chroma_db(
    chunks_750,
    f"{base_path}/rag/docs_750/chroma_db"
)
print("docs_750 Vector DB created successfully")


collection_1000 = build_chroma_db(
    chunks_1000,
    f"{base_path}/rag/docs_1000/chroma_db"
)
print("docs_1000 Vector DB created successfully")


collection_1500 = build_chroma_db(
    chunks_1500,
    f"{base_path}/rag/docs_1500/chroma_db"
)
print("docs_1500 Vector DB created successfully")

docs_100 Vector DB created successfully
docs_250 Vector DB created successfully
docs_500 Vector DB created successfully
docs_750 Vector DB created successfully
docs_1000 Vector DB created successfully
docs_1500 Vector DB created successfully


In [20]:
SYSTEM_PROMPT = """
You are maintaining a persistent concept-centric knowledge wiki.

A single source document may update MANY concept pages.

Your job is to extract meaningful concepts, entities, methods,
ideas, systems, techniques, relationships, and findings from
documents and create/update markdown pages.

IMPORTANT:
- Return ONLY valid JSON
- Each concept/entity should become its own page
- Use YAML frontmatter
- Use lowercase slugs for wiki links
- Use [[slug]] backlinks
- Preserve traceability to source documents
- Keep knowledge cumulative and append-only
- Prefer over-extraction rather than under-extraction

EXTRACT ALL POSSIBLE CONCEPTS INCLUDING:
- entities
- concepts
- methods
- techniques
- algorithms
- architectures
- models
- frameworks
- tools
- datasets
- metrics
- benchmarks
- tasks
- components
- systems
- modules
- workflows
- procedures
- strategies
- optimizations
- findings
- failures
- limitations
- issues
- technologies
- standards
- specifications
- relationships
- dependencies
- inputs
- outputs
- resources
- configurations
- evaluations
- experiments
- observations

EVERY meaningful concept should become a page if:
- it may be referenced later
- it may affect another entity
- it participates in relationships
- it may appear in future documents

RELATIONSHIP EXTRACTION

Extract relationships such as:
- uses
- depends_on
- related_to
- improves
- affects
- evaluated_by
- compared_with
- optimized_by
- implemented_by
- connected_to
- trained_on
- generates
- consumes
- references
- extends
- based_on
- replaces
- interacts_with

PAGE FORMAT

---
title: Example Concept
slug: example-concept
category: Concept
related: [related-concept]
sources: [document-001]
tags: [tag1, tag2]
last_updated: 2026-05-23
---

# Example Concept

## Summary
Concise technical summary.

## Technical Details
- Important information
- Key properties
- Constraints
- Behaviors

## Relationships

### Related
- [[related-concept]]

### Depends On
- [[dependency-concept]]

### Affects
- [[affected-concept]]

### References
- [[reference-concept]]

## Findings
- Important findings
- Observations
- Experimental notes

## Sources
- Source references

RULES

1. Create/update ONE PAGE PER CONCEPT.

2. Use concise but information-dense pages.

3. Use markdown lists and tables where appropriate.

4. Use wiki-style links:
   [[concept-slug]]

5. Create backlinks aggressively.

6. DO NOT generate narrative prose outside pages.

7. Return ONLY JSON.

8. Preserve terminology exactly where possible.

9. Prefer granular pages over large merged pages.

10. If unsure whether something deserves a page:
    CREATE THE PAGE.

JSON FORMAT

{
  "pages": [
    {
      "title": "Concept Title",
      "slug": "concept-title",
      "category": "Concept",
      "content": "FULL MARKDOWN PAGE"
    }
  ]
}
"""

In [21]:
def generate_pages(text):

    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": text
            }
        ],
        response_format={"type": "json_object"}
    )

    return json.loads(
        response.choices[0].message.content
    )

In [22]:
def update_index(index_file, slug, title):

    if not os.path.exists(index_file):

        with open(index_file, "w", encoding="utf-8") as f:
            f.write("# Concept Index\n\n")

    with open(index_file, "r", encoding="utf-8") as f:

        existing = f.read()

    entry = f"- [[{slug}]] — {title}"

    if entry not in existing:

        with open(index_file, "a", encoding="utf-8") as f:

            f.write(entry + "\n")

In [23]:
from datetime import datetime

def merge_content(existing, new):

    if not existing.strip():

        return new

    if new.strip() in existing:

        return existing

    return f"""
{existing}

---

# Incremental Update ({datetime.now().strftime('%Y-%m-%d')})

{new}
"""

In [26]:
def process_wikillm_documents(df, scale_name):

    pages_folder = f"{base_path}/wikillm/{scale_name}/pages"

    index_file = f"{base_path}/wikillm/{scale_name}/index.md"

    total_pages = 0

    for idx, row in df.iterrows():

        text = f'''
TITLE:
{row["title"]}

ABSTRACT:
{row["abstract"]}

FULL TEXT:
{row["full_text"]}
'''

        try:

            result = generate_pages(text)

            pages = result["pages"]

            total_pages += len(pages)

            for page in pages:

                slug = page["slug"]

                page_path = f"{pages_folder}/{slug}.md"

                new_content = page["content"]

                if os.path.exists(page_path):

                    with open(
                        page_path,
                        "r",
                        encoding="utf-8"
                    ) as f:

                        existing_content = f.read()

                else:

                    existing_content = ""

                merged = merge_content(
                    existing_content,
                    new_content
                )

                with open(
                    page_path,
                    "w",
                    encoding="utf-8"
                ) as f:

                    f.write(merged)

                update_index(
                    index_file,
                    slug,
                    page["title"]
                )

        except Exception as e:

            print("ERROR:", e)

    print("Total concept pages:", total_pages)

In [27]:
process_wikillm_documents(df_50, "docs_50")
print("docs_50 WikiLLM processing completed")

process_wikillm_documents(df_100, "docs_100")
print("docs_100 WikiLLM processing completed")

process_wikillm_documents(df_250, "docs_250")
print("docs_250 WikiLLM processing completed")

process_wikillm_documents(df_500, "docs_500")
print("docs_500 WikiLLM processing completed")

process_wikillm_documents(df_750, "docs_750")
print("docs_750 WikiLLM processing completed")

process_wikillm_documents(df_1000, "docs_1000")
print("docs_1000 WikiLLM processing completed")

process_wikillm_documents(df_1500, "docs_1500")
print("docs_1500 WikiLLM processing completed")

ERROR: Expecting ':' delimiter: line 19123 column 2 (char 37215)
ERROR: string indices must be integers, not 'str'
ERROR: Expecting ':' delimiter: line 10440 column 2 (char 38070)
Total concept pages: 309
docs_50 WikiLLM processing completed
ERROR: Expecting ':' delimiter: line 11978 column 1 (char 37901)
Total concept pages: 612
docs_100 WikiLLM processing completed
ERROR: Expecting ':' delimiter: line 8393 column 2 (char 39354)
ERROR: Expecting ':' delimiter: line 10625 column 2 (char 38675)
ERROR: Expecting ':' delimiter: line 13833 column 3 (char 61725)
ERROR: 'slug'
ERROR: string indices must be integers, not 'str'
ERROR: 'int' object is not subscriptable
ERROR: 'slug'
ERROR: 'slug'
ERROR: Expecting ':' delimiter: line 45680 column 2 (char 119030)
ERROR: Expecting ':' delimiter: line 61 column 30639 (char 41197)
Total concept pages: 1671
docs_250 WikiLLM processing completed
ERROR: string indices must be integers, not 'str'
ERROR: Expecting ':' delimiter: line 10377 column 1 (char

In [30]:
def build_wikillm_index(scale_name):

    index_path = f"{base_path}/wikillm/{scale_name}/index.md"

    with open(index_path, "r", encoding="utf-8") as f:

        lines = f.readlines()

    chroma_client = chromadb.PersistentClient(
        path=f"{base_path}/wikillm/{scale_name}/vector_db"
    )

    collection = chroma_client.get_or_create_collection(
        name="wikillm_index"
    )

    counter = 0

    for line in lines:

        if "[[" not in line:
            continue

        embedding = get_embedding(line)

        collection.add(
            ids=[str(counter)],
            documents=[line],
            embeddings=embedding
        )

        counter += 1

    print("Indexed concepts:", counter)

    return collection

In [31]:
wikillm_50 = build_wikillm_index("docs_50")
print("docs_50 WikiLLM index created")

wikillm_100 = build_wikillm_index("docs_100")
print("docs_100 WikiLLM index created")

wikillm_250 = build_wikillm_index("docs_250")
print("docs_250 WikiLLM index created")

wikillm_500 = build_wikillm_index("docs_500")
print("docs_500 WikiLLM index created")

wikillm_750 = build_wikillm_index("docs_750")
print("docs_750 WikiLLM index created")

wikillm_1000 = build_wikillm_index("docs_1000")
print("docs_1000 WikiLLM index created")

wikillm_1500 = build_wikillm_index("docs_1500")
print("docs_1500 WikiLLM index created")

Indexed concepts: 299
docs_50 WikiLLM index created
Indexed concepts: 565
docs_100 WikiLLM index created
Indexed concepts: 1325
docs_250 WikiLLM index created
Indexed concepts: 2475
docs_500 WikiLLM index created
Indexed concepts: 3578
docs_750 WikiLLM index created
Indexed concepts: 4720
docs_1000 WikiLLM index created
Indexed concepts: 6753
docs_1500 WikiLLM index created


In [33]:
query = "How does transformer attention work?"

query_embedding = get_embedding([query])[0]

results = wikillm_1500.query(
    query_embeddings=[query_embedding],
    n_results=5
)

results["documents"]

[['- [[attention-transformation]] — attention-transformation\n',
  '- [[transformer-architecture]] — transformer architecture\n',
  '- [[transformers]] — transformers (Transformer neural network architecture)\n',
  '- [[transformer-networks]] — transformer networks\n',
  '- [[transformer-fine-tuning]] — transformer-fine-tuning\n']]

In [35]:
# ============================================================
# COMPARE RAG vs WikiLLM METRICS
# ============================================================

def rag_metrics(chunks):

    total_chunks = len(chunks)

    total_chunk_chars = sum(
        len(chunk["text"])
        for chunk in chunks
    )

    avg_chunk_size = (
        total_chunk_chars / total_chunks
        if total_chunks > 0 else 0
    )

    return {
        "total_chunks": total_chunks,
        "total_chunk_characters": total_chunk_chars,
        "avg_chunk_size": round(avg_chunk_size, 2)
    }


# ============================================================
# GENERATE COMPARISON
# ============================================================

comparison = {}

datasets_info = [
    ("docs_50", chunks_50),
    ("docs_100", chunks_100),
    ("docs_250", chunks_250),
    ("docs_500", chunks_500),
    ("docs_750", chunks_750),
    ("docs_1000", chunks_1000),
    ("docs_1500", chunks_1500)
]

for scale_name, chunks in datasets_info:

    wiki_metrics = wikillm_metrics(scale_name)

    rag_metric = rag_metrics(chunks)

    comparison[scale_name] = {

        # WikiLLM
        "total_concept_pages":
            wiki_metrics["total_concept_pages"],

        "wikillm_total_characters":
            wiki_metrics["total_characters"],

        # RAG
        "total_chunks":
            rag_metric["total_chunks"],

        "rag_total_chunk_characters":
            rag_metric["total_chunk_characters"],

        "avg_chunk_size":
            rag_metric["avg_chunk_size"]
    }


# ============================================================
# PRINT COMPARISON
# ============================================================

comparison_df = pd.DataFrame(comparison).T

comparison_df

,total_concept_pages,wikillm_total_characters,total_chunks,rag_total_chunk_characters,avg_chunk_size
docs_50,296.0,347291.0,2576.0,1608445.0,624.40
docs_100,555.0,691819.0,5084.0,3192188.0,627.89
docs_250,1296.0,1731388.0,13074.0,8200188.0,627.21
docs_500,2390.0,3506069.0,26551.0,16582707.0,624.56
docs_750,3452.0,5297130.0,39551.0,24581399.0,621.51
docs_1000,4536.0,7145352.0,52097.0,32481672.0,623.48
docs_1500,6488.0,10517355.0,76461.0,47741801.0,624.39
